# Start here

This notebook checks that your FinUties key works against the live API at `https://data.finuties.com`.

## What
A four-step smoke test: load `FINUTIES_API_KEY`, hit `/health` and `/api/v1/status/p0`, list public recipes, then run one small COT query.

## Why this model
Before any chart, confirm auth, API reachability, and that a documented public endpoint returns rows. That is the repeatable baseline.

## How to rerun
1. Copy `notebooks/.env.example` to `notebooks/.env`.
2. Set `FINUTIES_API_KEY` from `POST https://data.finuties.com/api/v1/auth/sandbox` (the JSON `key` field).
3. Run every cell top to bottom.

Sandbox keys expire in about 72 hours. Longer-lived keys live in [FinUties settings](https://www.finuties.com/settings).

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


def _payload_as_of(payload) -> str | None:
    if not isinstance(payload, dict):
        return None
    for key in ("as_of", "generated_at", "updated_at", "report_date", "timestamp"):
        val = payload.get(key)
        if val:
            return str(val)
    return None


def _payload_source(payload) -> str | None:
    if not isinstance(payload, dict):
        return None
    for key in ("source", "source_family", "provider", "dataset"):
        val = payload.get(key)
        if val:
            return str(val)
    return None


def print_provenance(method: str, path: str, payload, params: dict | None = None) -> None:
    pulled_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    rows = normalize_rows(payload)
    if isinstance(payload, dict) and not rows:
        for key in ("domains", "recipes"):
            nested = payload.get(key)
            if isinstance(nested, list):
                rows = nested
                break

    row_count = len(rows) if isinstance(rows, list) else 0
    if row_count == 0:
        row_label = "empty"
    elif isinstance(payload, dict):
        expected = payload.get("total") or payload.get("count")
        if expected is not None and int(expected) != row_count:
            row_label = f"partial ({row_count} rows returned)"
        else:
            row_label = str(row_count)
    else:
        row_label = str(row_count)

    print("--- provenance ---")
    print(f"method: {method}")
    print(f"path: {path}")
    if params:
        print(f"params: {params}")
    print(f"pulled_at_utc: {pulled_at}")
    as_of = _payload_as_of(payload)
    if as_of:
        print(f"as_of: {as_of}")
    print(f"row_count: {row_label}")
    source = _payload_source(payload)
    if source:
        print(f"source: {source}")
    print("------------------")


## Health and P0 status

`/health` is unauthenticated. `/api/v1/status/p0` is the public coverage surface (filings, ownership, equities, positioning, rates, economics, calendar, system). `overall` can be green, yellow, or red — red means some domain is stale, not that the API is down.

In [ ]:
health = requests.get(f"{API_ORIGIN}/health", timeout=TIMEOUT_SECONDS)
health.raise_for_status()
health_payload = health.json()
print("health:", health_payload)
print_provenance("GET", "/health", health_payload)
assert health_payload.get("status") == "online", health_payload

p0 = finuties_get("/api/v1/status/p0")
print_provenance("GET", "/api/v1/status/p0", p0)
require_frame(pd.DataFrame(p0.get("domains") or []), ["domain", "status"], min_rows=1)
print("p0 overall:", p0.get("overall"))
print("p0 summary:", p0.get("summary"))
pd.DataFrame(p0["domains"])[["domain", "status", "budget_hours"]]

## Recipes, then one query

`GET /api/v1/recipes` is the public job list. The tiny query is wheat COT facts (`GET /api/v1/cftc/legacy_futures-facts`) — verified live, not a guessed `/api/v1/data/...` path.

In [ ]:
recipes_payload = finuties_get("/api/v1/recipes")
print_provenance("GET", "/api/v1/recipes", recipes_payload)
recipes = pd.DataFrame(recipes_payload.get("recipes") or [])
require_frame(recipes, ["id", "title", "path"], min_rows=1)
print(f"recipes: {len(recipes)}")

cot_params = {"commodity": "WHEAT", "limit": 5}
cot_payload = finuties_get("/api/v1/cftc/legacy_futures-facts", cot_params)
print_provenance("GET", "/api/v1/cftc/legacy_futures-facts", cot_payload, cot_params)
cot = pd.DataFrame(normalize_rows(cot_payload))
require_frame(cot, ["commodity_name", "report_date_as_yyyy_mm_dd"], min_rows=1)
assert (cot["commodity_name"] == "WHEAT").any(), "Expected WHEAT rows from the COT query"
print(f"COT sample rows: {len(cot)}")
recipes[["id", "title", "path"]]

## Caveats

- A 200 from `/health` does not mean every dataset is fresh. Read `status/p0`.
- Sandbox keys are short-lived and rate-limited.
- Do not commit `notebooks/.env`.

Next: `money_flow/latest_cot_data_example.ipynb`, then commodities, macro, equity flows, and risk models.